# Leaving for a Living: Measuring Labor Inequality and Inaccessibility through the Impact of Macroeconomic and Regional Factors on the Financial Mobility of OFWs

**SDGs addressed**

- **SDG 1 — No Poverty:** Poverty Alleviation, Income Equality, Social Protection
- **SDG 8 — Decent Work and Economic Growth:** Job Creation, Fair Wages, Economic Development

**Data source:** PSA Survey on Overseas Filipinos (SOF) 2024, Public Use File.

## Research questions

1. What drives OFWs to push towards overseas contractual work, and what are the initial reasons for migrating abroad given local employment conditions?
2. What individual characteristics (gender, marital status, destination country, contract status) of Filipino workers predict their ability to maintain personal savings abroad, in comparison to remitting near 100% of monthly earnings to their family?
3. To what extent does an OFW's regional geographical location predict their chosen remittance transfer mode (Formal Banks vs. Door-to-door vs. OTC)?

## Hypotheses

**RQ1**
- H0: Pre-departure domestic employment characteristics and regional demographics have no statistical significance on why OFWs migrate to contractual overseas work.
- H1: OFWs come disproportionately from lower-wage domestic occupations and from outside the capital region, which gives statistical significance to migrating for contractual labor.

**RQ2**
- H0: An individual's demographic characteristics have no statistically significant relationship with an OFW's allocation of monthly income with regards to personal savings.
- H1: Having a higher level of educational attainment and a secure contract status make it more likely an OFW maintains personal savings abroad rather than remitting nearly all earnings to their family.

**RQ3**
- H0: The choice of remittance transfer mode is statistically independent of regional location.
- H1: Recipient households in island and rural areas rely more on OTC or door-to-door transfer compared to urbanized locations.

## Objectives

1. Identify the different factors that push Filipinos to work abroad compared to finding work domestically.
2. Model the determinants of remittance/savings allocation using: **demographic traits, human capital, contract status, and foreign destination.**
3. Classify the primary remittance channel of OFWs using geographical location and socioeconomic status, to understand the underlying factors influencing financial/banking access.

This notebook runs end-to-end: it loads the raw PUF, answers each research question with a clustered hypothesis test, trains a machine-learning classifier for objectives 2 and 3, and writes figures/tables to `outputs/`.

> **Flag before you read further:** RQ2 cites item **C18** of the questionnaire directly. Cross-checking `SOF 2024 Questionnaire.pdf` against the data dictionary confirms C18 was asked (personal savings vs. remittance allocation of monthly compensation abroad) but was **not included in this Public Use File** — the variable list jumps from C17 straight to C19. Section 2 below documents this and the substitute measure used instead.

## 1. Imports, paths, and recoding tables

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from patsy import dmatrix
from scipy.stats import chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
	accuracy_score,
	balanced_accuracy_score,
	confusion_matrix,
	f1_score,
	roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.discrete.discrete_model import MNLogit
import statsmodels.formula.api as smf

from IPython.display import Image, display

ROOT = Path.cwd()
DATA_PATH = ROOT / "PHL-PSA-SOF-2024-PUF 2" / "SOF PUF 2024.CSV"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

In [ ]:
OCCUPATION_RANGES = {
	"Armed forces": range(1, 4),
	"Managers": range(11, 15),
	"Professionals": range(21, 27),
	"Technicians": range(31, 36),
	"Clerical support": range(41, 45),
	"Services and sales": range(51, 55),
	"Agriculture, forestry, fishery": range(61, 64),
	"Craft and trades": range(71, 76),
	"Plant and machine operators": range(81, 84),
	"Elementary occupations": range(91, 97),
	"Non-gainful": [9],
}
OCCUPATION_LABELS = {
	code: label for label, codes in OCCUPATION_RANGES.items() for code in codes
}
EDU_GROUPS = {
	0: "Below junior high", 1: "Below junior high", 2: "Below junior high",
	3: "Below junior high", 4: "High school", 5: "High school",
	6: "High school", 7: "Post-secondary / TVET", 8: "Post-secondary / TVET",
	9: "College or higher", 10: "College or higher", 11: "College or higher",
}
EDU_ORDER = ["Below junior high", "High school", "Post-secondary / TVET", "College or higher"]

MARITAL_GROUPS = {
	1: "Single", 2: "Married", 3: "Married",
	4: "Previously married", 5: "Previously married",
	6: "Previously married", 7: "Previously married",
}
MARITAL_ORDER = ["Single", "Married", "Previously married"]

DESTINATION_RANGES = {
	"Africa": [100],
	"East Asia": [211, 212, 213, 214],
	"Southeast / South Central Asia": [221, 222, 223],
	"Western Asia (Middle East)": [231, 232, 233, 234, 235],
	"Australia": [300],
	"Europe": [400],
	"Americas": [500],
	"Other": [600],
}
DESTINATION_LABELS = {
	code: label for label, codes in DESTINATION_RANGES.items() for code in codes
}
DESTINATION_ORDER = list(DESTINATION_RANGES.keys())

REMIT_MODE_GROUPS = {
	1: "Formal Bank", 2: "OTC", 3: "OTC", 4: "Door-to-door", 5: "OTC", 6: "OTC",
}
REMIT_MODE_ORDER = ["Formal Bank", "Door-to-door", "OTC"]

## 2. Data loading and labeling

`region_group` collapses the 17 PSA regions into NCR / Luzon / Visayas / Mindanao island groups. `load_data` reads the raw PUF, coerces the numeric codes, and derives every analysis-ready column used below: contract status, occupation group, education group, marital-status group, destination region, urban/rural, any-savings outcome, and the 3-category remittance mode (Formal Bank / Door-to-door / OTC).

In [ ]:
def region_group(code: float) -> str | float:
	if pd.isna(code):
		return np.nan
	code = int(code)
	if code == 13:
		return "NCR"
	if code in {14, 1, 2, 3, 4, 17, 5}:
		return "Luzon"
	if code in {6, 7, 8}:
		return "Visayas"
	if code in {9, 10, 11, 12, 16, 19}:
		return "Mindanao"
	return np.nan

In [ ]:
def load_data() -> pd.DataFrame:
	data = pd.read_csv(DATA_PATH, dtype=str, encoding="utf-8-sig")
	numeric = [
		"RQ3_SEX", "RQ4_AGE", "RQ7_MSTAT", "RQ8_HGRADE", "RQ9_2DGT", "RQ10_REASON",
		"RQ12_CTRY", "RQ17_AVEMINC", "RQ26_MODE", "RQ273P_SAVINGS", "RP6M_IND",
		"RREG", "RURB2020", "RSWGT",
	]
	for column in numeric:
		data[column] = pd.to_numeric(data[column].str.strip(), errors="coerce")

	data["contract"] = data["RQ10_REASON"].eq(1)
	data["contract_num"] = data["contract"].astype(int)
	data["occupation"] = data["RQ9_2DGT"].map(
		lambda value: OCCUPATION_LABELS.get(int(value)) if pd.notna(value) else np.nan
	)
	data["education"] = data["RQ8_HGRADE"].map(EDU_GROUPS)
	data["region_group"] = data["RREG"].map(region_group)
	data["capital_origin"] = data["region_group"].map(
		lambda value: "NCR" if value == "NCR" else "Outside NCR" if pd.notna(value) else np.nan
	)
	data["urban_rural"] = data["RURB2020"].map({1: "Urban", 2: "Rural"})
	data["sex"] = data["RQ3_SEX"].map({1: "Male", 2: "Female"})
	data["marital_status"] = data["RQ7_MSTAT"].map(MARITAL_GROUPS)
	data["destination_region"] = data["RQ12_CTRY"].map(
		lambda value: DESTINATION_LABELS.get(int(value)) if pd.notna(value) else np.nan
	)
	data["savings_pct"] = data["RQ273P_SAVINGS"].where(data["RQ273P_SAVINGS"].between(0, 100))
	data["any_savings"] = data["savings_pct"].gt(0).where(data["savings_pct"].notna())
	data["any_savings_num"] = data["any_savings"].astype(float)
	data["mode"] = data["RQ26_MODE"].map(
		lambda value: REMIT_MODE_GROUPS.get(int(value)) if pd.notna(value) else np.nan
	)
	data["mode_code"] = data["mode"].map({"Formal Bank": 0, "Door-to-door": 1, "OTC": 2})
	return data

In [ ]:
data = load_data()
print(f"Records: {len(data):,}; households: {data['RHHNUM'].nunique():,}")
data.head()

### A note on what the questionnaire actually asks (cross-checked against `SOF 2024 Questionnaire.pdf` and the data dictionary)

Re-reading the raw SOF Form 1 questionnaire and `sof_2024_metadata(dictionary).xlsx` against the released PUF surfaces a few things that materially affect how the research questions can be answered:

1. **C18 — the field RQ2 literally cites in parentheses — is not in the Public Use File.** The questionnaire's C18 asks *"Of the amount received from his/her monthly compensation, how much did [the OFW] allocate on the following: remittance | consumption | savings | others, specify | Don't know"* — i.e., exactly "personal savings abroad vs. remitting nearly 100% home." The variable list jumps straight from `RQ17_AVEMINC` (C17) to `RQ19_NOMONTH` (C19); `RQ18_*` does not exist anywhere in the dictionary or the CSV header. The closest available substitute is `RQ273P_SAVINGS` (C27), which only measures how the *already-remitted* money is allocated by the receiving family, not what the OFW kept abroad.
2. **C24–C31 (remittance amount, mode, spending shares) follow a questionnaire skip pattern, not random missingness.** They are only asked for respondents flagged by `RP6M_IND == 1` ("included in past six months"); `RP6M_IND == 2` is overwhelmingly Tourists/Students/Others (`RQ10_REASON` 4/5/8) who were never asked these items. That is why the analysis sample for RQ2/RQ3 drops from {len(data):,} to roughly 2,600–2,700 — it is a designed skip, not attrition.
3. **Occupation, marital status, and education are explicitly pre-departure.** C9 is *"usual occupation in the Philippines before he/she (last) left"*, and C7/C8 are marital status / highest grade *"as of his/her (last) departure."* These are genuinely pre-departure domestic characteristics, which is exactly what RQ1's hypothesis requires — not a weak proxy.
4. **The total cash remittance field (`RQ25_CASHAMT`, C25) is a fixed six-month window (April–September 2024)**, summed from six monthly entries. It does not line up with `RQ19_NOMONTH` (months worked over the full Oct 2019–Sept 2024 reference period), so a remittance-to-income ratio built from these two fields is not interpretable — confirmed by checking the questionnaire wording directly, not just inferred from odd descriptive statistics.

The cell below checks the numeric fields used throughout this notebook for PSA's typical sentinel "not reported" codes (e.g. `9999999` for income, `99999999` for cash amounts) — none are present in this release, so no additional sentinel-filtering is required beyond the `between()` bounds already applied in `load_data`.

In [ ]:
sentinel_checks = {
	"RQ17_AVEMINC (income)": (data["RQ17_AVEMINC"], 9999999),
	"RQ4_AGE (age)": (data["RQ4_AGE"], 999),
}
for label, (series, sentinel) in sentinel_checks.items():
	print(f"{label}: max={series.max():,.0f}, sentinel {sentinel} present={bool((series == sentinel).any())}")

print()
print("RP6M_IND skip pattern (why RQ2/RQ3 have a smaller N than RQ1):")
display(pd.crosstab(data["RQ10_REASON"], data["RP6M_IND"], dropna=False))

## 3. Statistical helper functions

- `wmean` — survey-weighted mean.
- `chi_result` — crude (unadjusted) chi-square association plus Cramer's V effect size.
- `clustered_ols_terms` / `term_result` — linear-probability models with household-clustered robust standard errors, used as the primary hypothesis tests so that repeated household records don't understate uncertainty.
- `clustered_multinomial_tests` — a clustered multinomial logit for the 3-category remittance-mode outcome in RQ3.

In [ ]:
def wmean(values: pd.Series, weights: pd.Series) -> float:
	valid = values.notna() & weights.notna() & weights.gt(0)
	return float(np.average(values[valid], weights=weights[valid])) if valid.any() else np.nan


def chi_result(table: pd.DataFrame) -> dict[str, float | int]:
	statistic, pvalue, dof, expected = chi2_contingency(table)
	dimension = min(table.shape[0] - 1, table.shape[1] - 1)
	n = int(table.to_numpy().sum())
	return {
		"n": n, "chi2": float(statistic), "df": int(dof), "p": float(pvalue),
		"cramers_v": float(np.sqrt(statistic / (n * dimension))) if dimension else 0.0,
		"min_expected": float(expected.min()),
	}


def clustered_ols_terms(formula: str, data: pd.DataFrame) -> tuple[object, pd.DataFrame]:
	fitted = smf.ols(formula, data=data).fit(
		cov_type="cluster", cov_kwds={"groups": data["RHHNUM"]}
	)
	return fitted, fitted.wald_test_terms(scalar=True).table


def term_result(terms: pd.DataFrame, term: str) -> dict[str, float | int]:
	row = terms.loc[term]
	return {
		"statistic": float(row["statistic"]),
		"pvalue": float(row["pvalue"]),
		"df": int(row["df_constraint"]),
	}


def pvalue(value: float) -> str:
	return "< 0.001" if value < 0.001 else f"= {value:.3f}"

In [ ]:
def clustered_multinomial_tests(data: pd.DataFrame) -> tuple[dict[str, dict[str, float | int]], object]:
	design = dmatrix("1 + C(urban_rural) + C(region_group)", data, return_type="dataframe")
	fitted = MNLogit(data["mode_code"].astype(int), design).fit(
		disp=False,
		maxiter=200,
		cov_type="cluster",
		cov_kwds={"groups": data["RHHNUM"]},
	)

	def joint_test(term: str) -> dict[str, float | int]:
		rows = [index for index, name in enumerate(design.columns) if term in name]
		n_equations = fitted.params.shape[1]
		indices = [
			np.ravel_multi_index((row, equation), fitted.params.shape, order="F")
			for row in rows
			for equation in range(n_equations)
		]
		restriction = np.eye(fitted.params.size)[indices]
		test = fitted.wald_test(restriction, scalar=True)
		return {
			"statistic": float(test.statistic),
			"pvalue": float(test.pvalue),
			"df": len(indices),
		}

	results = {
		"Urban/rural location": joint_test("C(urban_rural)"),
		"Island group": joint_test("C(region_group)"),
		"All location terms": joint_test("C("),
	}
	return results, fitted

## 4. Shared plotting helper

`feature_importance_plot` pulls feature names back out of a fitted `ColumnTransformer` + `RandomForestClassifier` pipeline and renders a horizontal bar chart of the top drivers — used for both machine-learning models below.

In [ ]:
def feature_importance_plot(pipeline: Pipeline, title: str, filename: str, top_n: int = 15) -> pd.Series:
	names = pipeline.named_steps["preprocess"].get_feature_names_out()
	importances = pipeline.named_steps["model"].feature_importances_
	series = pd.Series(importances, index=names).sort_values(ascending=True).tail(top_n)
	fig, ax = plt.subplots(figsize=(8, 0.35 * len(series) + 1.5))
	ax.barh(series.index, series.values, color="#176b63")
	ax.set(title=title, xlabel="Random forest importance")
	fig.tight_layout()
	fig.savefig(OUT / filename, dpi=160, bbox_inches="tight")
	plt.close(fig)
	return series.sort_values(ascending=False)


def confusion_matrix_plot(y_true, y_pred, labels: list[str], title: str, filename: str) -> pd.DataFrame:
	matrix = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
	table = pd.DataFrame(matrix, index=[f"Actual: {l}" for l in labels], columns=[f"Predicted: {l}" for l in labels])
	fig, ax = plt.subplots(figsize=(1.6 * len(labels) + 2, 1.6 * len(labels) + 1))
	im = ax.imshow(matrix, cmap="Blues")
	ax.set_xticks(range(len(labels)), labels, rotation=20, ha="right")
	ax.set_yticks(range(len(labels)), labels)
	ax.set(title=title, xlabel="Predicted", ylabel="Actual")
	for i in range(len(labels)):
		for j in range(len(labels)):
			ax.text(j, i, f"{matrix[i, j]:,}", ha="center", va="center",
				color="white" if matrix[i, j] > matrix.max() / 2 else "black")
	fig.colorbar(im, ax=ax, shrink=0.8, label="Count")
	fig.tight_layout()
	fig.savefig(OUT / filename, dpi=160, bbox_inches="tight")
	plt.close(fig)
	return table

## 5. RQ1 — What drives OFWs toward contractual overseas work?

**H0:** Pre-departure domestic employment characteristics and regional demographics have no statistical significance on why OFWs migrate to contractual overseas work.
**H1:** OFWs come disproportionately from lower-wage domestic occupations and from outside the capital region, which gives statistical significance to migrating for contractual labor.

Operationalization: contract-OFW status (`RQ10_REASON = 1`) among people already represented in the overseas-Filipino survey, compared across usual occupation and NCR/outside-NCR household origin. Per the questionnaire, both are genuinely pre-departure: C9 asks occupation "before he/she (last) left," and household region is the pre-migration residence. The file still does not include non-migrants or domestic wages, so occupation is only a prior-work proxy for wage level and region is only a rough inequality proxy — this tests association among surveyed OFWs, not a true push-factor model against a non-migrant baseline.

In [ ]:
h1_data = data.dropna(subset=["occupation", "capital_origin"]).copy()
h1_model, h1_terms = clustered_ols_terms(
	"contract_num ~ C(occupation) + C(capital_origin)", h1_data
)
h1_occupation = term_result(h1_terms, "C(occupation)")
h1_location = term_result(h1_terms, "C(capital_origin)")
h1_crude_occupation = chi_result(pd.crosstab(h1_data["occupation"], h1_data["contract"]))
h1_crude_location = chi_result(pd.crosstab(h1_data["capital_origin"], h1_data["contract"]))

contract_share = 100 * wmean(data["contract"].astype(float), data["RSWGT"])
h1_occ_df, h1_occ_stat, h1_occ_p = h1_occupation["df"], h1_occupation["statistic"], h1_occupation["pvalue"]
h1_loc_df, h1_loc_stat, h1_loc_p = h1_location["df"], h1_location["statistic"], h1_location["pvalue"]
print(f"Weighted contract-OFW share: {contract_share:.1f}%")
print(f"Occupation (adj. for NCR origin): Wald F({h1_occ_df})={h1_occ_stat:.2f}, p {pvalue(h1_occ_p)}")
print(f"NCR origin (adj. for occupation): Wald F({h1_loc_df})={h1_loc_stat:.2f}, p {pvalue(h1_loc_p)}")

In [ ]:
def plot_h1(data: pd.DataFrame) -> None:
	fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
	valid = data.dropna(subset=["occupation", "RSWGT"])
	occupation_rates = valid.groupby("occupation", observed=True).apply(
		lambda group: 100 * wmean(group["contract"].astype(float), group["RSWGT"]),
		include_groups=False,
	).sort_values()
	axes[0].barh(occupation_rates.index, occupation_rates.values, color="#176b63")
	axes[0].set(title="Contract OFW share by prior occupation", xlabel="Weighted share (%)", xlim=(0, 100))
	capital = data.dropna(subset=["region_group", "RSWGT"]).copy()
	capital["origin"] = np.where(capital["region_group"].eq("NCR"), "NCR", "Outside NCR")
	rates = capital.groupby("origin").apply(
		lambda group: 100 * wmean(group["contract"].astype(float), group["RSWGT"]),
		include_groups=False,
	).reindex(["NCR", "Outside NCR"])
	axes[1].bar(rates.index, rates.values, color="#dd8a35")
	axes[1].set(title="Contract OFW share by household region", ylabel="Weighted share (%)", ylim=(0, 100))
	fig.suptitle("Contract status and observed origins", fontsize=16, weight="bold")
	fig.tight_layout()
	fig.savefig(OUT / "h1_contract_status.png", dpi=180, bbox_inches="tight")
	plt.close(fig)


plot_h1(data)
display(Image(filename=str(OUT / "h1_contract_status.png")))

## 6. RQ2 — Demographics and maintaining personal savings abroad

**H0:** An individual's demographic characteristics have no statistically significant relationship with an OFW's allocation of monthly income with regards to personal savings.
**H1:** Having a higher level of educational attainment and a secure contract status make it more likely an OFW maintains personal savings rather than remitting nearly all earnings to their family.

The questionnaire's C18 asks exactly this — how monthly compensation earned abroad is allocated between remittance, consumption, personal savings, and others — but `RQ18_*` is absent from the released PUF entirely (the dictionary jumps straight from C17/`RQ17_AVEMINC` to C19/`RQ19_NOMONTH`). The total-cash-remittance field also can't substitute: `RQ25_CASHAMT` (C25) is a fixed six-month sum (April–September 2024) that doesn't line up with `RQ19_NOMONTH` (months worked over the full five-year reference period), so a remittance-to-income ratio isn't interpretable either. The closest available proxy, given what *is* released, is whether any share of what *is* remitted is allocated to savings by the receiving family (`RQ273P_SAVINGS > 0`, `any_savings`, from C27) versus all of it going to consumption/gifts. This is tested against the predictors named in the research question — sex, marital status, destination country (grouped into regions), and contract status — with household-clustered robust standard errors.

In [ ]:
savings = data.dropna(
	subset=["any_savings_num", "sex", "marital_status", "destination_region", "contract_num"]
).copy()
h2_model, h2_terms = clustered_ols_terms(
	"any_savings_num ~ C(sex) + C(marital_status) + C(destination_region) + contract_num", savings
)
h2_sex = term_result(h2_terms, "C(sex)")
h2_marital = term_result(h2_terms, "C(marital_status)")
h2_destination = term_result(h2_terms, "C(destination_region)")
h2_contract = term_result(h2_terms, "contract_num")
h2_crude_sex = chi_result(pd.crosstab(savings["sex"], savings["any_savings"]))
h2_crude_marital = chi_result(pd.crosstab(savings["marital_status"], savings["any_savings"]))
h2_crude_destination = chi_result(pd.crosstab(savings["destination_region"], savings["any_savings"]))
h2_crude_contract = chi_result(pd.crosstab(savings["contract"], savings["any_savings"]))

any_savings_share = 100 * wmean(data["any_savings"].astype(float), data["RSWGT"])
print(f"Any-savings share: {any_savings_share:.1f}% (N={len(savings):,})")
for label, term in [("Sex", h2_sex), ("Marital status", h2_marital), ("Destination region", h2_destination), ("Contract", h2_contract)]:
	df, stat, p = term["df"], term["statistic"], term["pvalue"]
	print(f"{label}: Wald F({df})={stat:.2f}, p {pvalue(p)}")

In [ ]:
def plot_h2(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
	valid = data.dropna(subset=["savings_pct", "marital_status", "destination_region", "RSWGT"]).copy()
	valid["marital_status"] = pd.Categorical(valid["marital_status"], MARITAL_ORDER, ordered=True)
	marital = valid.groupby("marital_status", observed=True).apply(
		lambda group: pd.Series({
			"weighted_mean_pct": wmean(group["savings_pct"], group["RSWGT"]),
			"n": len(group),
		}), include_groups=False,
	).reset_index()
	valid["destination_region"] = pd.Categorical(valid["destination_region"], DESTINATION_ORDER, ordered=True)
	destination = valid.groupby("destination_region", observed=True).apply(
		lambda group: pd.Series({
			"weighted_mean_pct": wmean(group["savings_pct"], group["RSWGT"]),
			"n": len(group),
		}), include_groups=False,
	).reset_index()
	fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
	axes[0].bar(marital["marital_status"].astype(str), marital["weighted_mean_pct"], color="#3278a3")
	axes[0].set(title="Savings share by marital status", xlabel="Marital status", ylabel="Weighted mean savings (%)")
	axes[0].tick_params(axis="x", rotation=12)
	axes[1].bar(destination["destination_region"].astype(str), destination["weighted_mean_pct"], color="#c85442")
	axes[1].set(title="Savings share by destination region", ylabel="Weighted mean savings (%)")
	axes[1].tick_params(axis="x", rotation=35)
	for label in axes[1].get_xticklabels():
		label.set_ha("right")
	fig.suptitle("Reported allocation of remittances to savings", fontsize=16, weight="bold")
	fig.tight_layout()
	fig.savefig(OUT / "h2_savings_allocation.png", dpi=180, bbox_inches="tight")
	plt.close(fig)
	return marital, destination


marital_summary, destination_summary = plot_h2(data)
display(Image(filename=str(OUT / "h2_savings_allocation.png")))

### Objective 2 — modeling the determinants of savings allocation

A machine-learning classifier predicts `any_savings` from the four determinant groups named in the objective: **demographic traits** (age, sex, marital status), **human capital** (highest grade completed), **contract** status, and **foreign destination** (destination region). A logistic regression and a random forest are compared against a weighted-majority baseline.

Evaluation uses a household-grouped, stratified 80/20 split (no household appears in both train and test) and survey weights for both fitting and scoring, so the headline numbers reflect population-representative performance rather than raw-row performance.

In [ ]:
def fit_savings_model(data: pd.DataFrame):
	sample = data.dropna(subset=["savings_pct", "RSWGT"]).copy()
	sample = sample.loc[sample["RSWGT"].gt(0)]
	numeric = ["RQ4_AGE", "RQ8_HGRADE"]
	categorical = ["sex", "marital_status", "contract", "destination_region"]
	X = sample[numeric + categorical].copy()
	for column in categorical:
		X[column] = X[column].astype("object")
	y = sample["savings_pct"].gt(0).astype(int)
	weights = sample["RSWGT"].astype(float)
	transform = ColumnTransformer([
		("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
		("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
	])
	train_index, test_index = next(
		StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=132).split(
			X, y, groups=sample["RHHNUM"]
		)
	)
	assert set(sample.iloc[train_index]["RHHNUM"]).isdisjoint(sample.iloc[test_index]["RHHNUM"])
	X_train, X_test = X.iloc[train_index], X.iloc[test_index]
	y_train, y_test = y.iloc[train_index], y.iloc[test_index]
	w_train, w_test = weights.iloc[train_index], weights.iloc[test_index]
	estimators = {
		"Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
		"Random forest": RandomForestClassifier(
			n_estimators=300, min_samples_leaf=8, class_weight="balanced_subsample",
			random_state=132, n_jobs=-1,
		),
	}
	results: dict[str, dict[str, float]] = {}
	pipelines: dict[str, Pipeline] = {}
	predictions: dict[str, np.ndarray] = {}
	for name, estimator in estimators.items():
		model = Pipeline([("preprocess", transform), ("model", estimator)])
		model.fit(X_train, y_train, model__sample_weight=w_train / w_train.mean())
		probabilities = model.predict_proba(X_test)[:, 1]
		prediction = (probabilities >= 0.5).astype(int)
		results[name] = {
			"n_train": len(y_train), "n_test": len(y_test),
			"weighted_accuracy": accuracy_score(y_test, prediction, sample_weight=w_test),
			"weighted_balanced_accuracy": balanced_accuracy_score(y_test, prediction, sample_weight=w_test),
			"weighted_roc_auc": roc_auc_score(y_test, probabilities, sample_weight=w_test),
			"f1_macro": f1_score(y_test, prediction, average="macro"),
		}
		pipelines[name] = model
		predictions[name] = prediction
	prevalence = wmean(y_train.astype(float), w_train)
	results["Weighted majority baseline"] = {
		"n_train": len(y_train), "n_test": len(y_test),
		"weighted_accuracy": max(prevalence, 1 - prevalence),
		"weighted_balanced_accuracy": 0.5, "weighted_roc_auc": 0.5, "f1_macro": np.nan,
	}
	return results, len(sample), pipelines, (y_test, predictions)


savings_models, savings_model_n, savings_pipelines, (savings_y_test, savings_predictions) = fit_savings_model(data)
savings_model_table = pd.DataFrame(savings_models).T
savings_model_table.to_csv(OUT / "model_metrics_savings.csv", index_label="model")
display(savings_model_table)

In [ ]:
savings_importance = feature_importance_plot(
	savings_pipelines["Random forest"],
	"RQ2 model: what predicts any reported savings?",
	"rq2_feature_importance.png",
)
display(Image(filename=str(OUT / "rq2_feature_importance.png")))
savings_importance.head(10)

In [ ]:
savings_confusion = confusion_matrix_plot(
	savings_y_test, savings_predictions["Random forest"],
	 labels=["No savings", "Any savings"],
	title="RQ2 model: confusion matrix (random forest, held-out)",
	filename="rq2_confusion_matrix.png",
)
display(Image(filename=str(OUT / "rq2_confusion_matrix.png")))
savings_confusion

## 7. RQ3 — Remittance transfer mode and regional location

**H0:** The choice of remittance transfer mode is statistically independent of regional location.
**H1:** Recipient households in island and rural areas rely more on OTC or door-to-door transfer compared to urbanized locations.

`RQ26_MODE` is recoded into three channels matching the hypothesis — **Formal Bank** (bank), **Door-to-door** (door-to-door carrier), and **OTC** (agency/local office, friends/co-workers, money-transfer services such as Western Union/MoneyGram, and other modes). Geography available in the PUF is household region (grouped into island groups) and urban/rural classification, not province. The clustered multinomial model tests the joint location terms across the three channels.

In [ ]:
modes = data.dropna(subset=["mode", "mode_code", "region_group", "urban_rural"]).copy()
h3_tests, h3_model = clustered_multinomial_tests(modes)
h3_urban = chi_result(pd.crosstab(modes["urban_rural"], modes["mode"]))
h3_region = chi_result(pd.crosstab(modes["region_group"], modes["mode"]))
door_count = int((data["mode"] == "Door-to-door").sum())

for label, key in [("Urban/rural", "Urban/rural location"), ("Island group", "Island group"), ("All location terms", "All location terms")]:
	term = h3_tests[key]
	df, stat, p = term["df"], term["statistic"], term["pvalue"]
	print(f"{label}: Wald chi-square({df})={stat:.2f}, p {pvalue(p)}")
print(f"Door-to-door observations: {door_count} (too rare for reliable inference or ML)")

In [ ]:
def plot_h3(data: pd.DataFrame) -> None:
	valid = data.dropna(subset=["mode", "urban_rural", "region_group"])
	fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
	for axis, location, title in [
		(axes[0], "urban_rural", "Mode by urban/rural household"),
		(axes[1], "region_group", "Mode by island group"),
	]:
		share = pd.crosstab(valid[location], valid["mode"], normalize="index").mul(100)
		share = share.reindex(
			index=["Urban", "Rural"] if location == "urban_rural" else ["Luzon", "Visayas", "Mindanao", "NCR"],
			columns=REMIT_MODE_ORDER,
			fill_value=0,
		)
		share.plot(
			kind="bar", stacked=True, ax=axis, color=["#176b63", "#dd8a35", "#6c7780"]
		)
		axis.set(title=title, ylabel="Respondents (%)", xlabel="")
		axis.tick_params(axis="x", rotation=15)
		axis.legend().remove()
	handles, labels = axes[0].get_legend_handles_labels()
	fig.legend(handles, labels, title="Reported mode", loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.02), frameon=False)
	fig.suptitle("Remittance channel and household location", fontsize=16, weight="bold")
	fig.tight_layout(rect=[0, 0.09, 1, 0.94])
	fig.savefig(OUT / "h3_mode_by_location.png", dpi=180, bbox_inches="tight")
	plt.close(fig)


plot_h3(data)
display(Image(filename=str(OUT / "h3_mode_by_location.png")))

### Objective 3 — classifying the primary remittance channel

A second machine-learning classifier directly targets objective 3 — classifying the primary remittance channel of OFWs by geographical location and socioeconomic status. Predictors are explicitly geographic (island group, urban/rural) and socioeconomic (monthly income, education, age).

Door-to-door is excluded from this classifier (only 3 observations in the whole file — too few to split into train/test folds, let alone learn from), so this model is a binary **Formal Bank vs. OTC** classifier. That restriction is itself a finding: in this file, OTC/money-transfer services have effectively displaced door-to-door carriers as the non-bank channel, which the earlier hypothesis test already flags.

In [ ]:
def fit_mode_model(data: pd.DataFrame):
	sample = data.dropna(subset=["mode", "RSWGT"]).copy()
	sample = sample.loc[sample["mode"].isin(["Formal Bank", "OTC"])]
	sample = sample.loc[sample["RSWGT"].gt(0)]
	numeric = ["RQ4_AGE", "RQ17_AVEMINC"]
	categorical = ["region_group", "urban_rural", "education"]
	X = sample[numeric + categorical].copy()
	for column in categorical:
		X[column] = X[column].astype("object")
	y = sample["mode"].eq("OTC").astype(int)
	weights = sample["RSWGT"].astype(float)
	transform = ColumnTransformer([
		("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
		("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
	])
	train_index, test_index = next(
		StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=132).split(
			X, y, groups=sample["RHHNUM"]
		)
	)
	assert set(sample.iloc[train_index]["RHHNUM"]).isdisjoint(sample.iloc[test_index]["RHHNUM"])
	X_train, X_test = X.iloc[train_index], X.iloc[test_index]
	y_train, y_test = y.iloc[train_index], y.iloc[test_index]
	w_train, w_test = weights.iloc[train_index], weights.iloc[test_index]
	estimators = {
		"Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
		"Random forest": RandomForestClassifier(
			n_estimators=300, min_samples_leaf=8, class_weight="balanced_subsample",
			random_state=132, n_jobs=-1,
		),
	}
	results: dict[str, dict[str, float]] = {}
	pipelines: dict[str, Pipeline] = {}
	predictions: dict[str, np.ndarray] = {}
	for name, estimator in estimators.items():
		model = Pipeline([("preprocess", transform), ("model", estimator)])
		model.fit(X_train, y_train, model__sample_weight=w_train / w_train.mean())
		probabilities = model.predict_proba(X_test)[:, 1]
		prediction = (probabilities >= 0.5).astype(int)
		results[name] = {
			"n_train": len(y_train), "n_test": len(y_test),
			"weighted_accuracy": accuracy_score(y_test, prediction, sample_weight=w_test),
			"weighted_balanced_accuracy": balanced_accuracy_score(y_test, prediction, sample_weight=w_test),
			"weighted_roc_auc": roc_auc_score(y_test, probabilities, sample_weight=w_test),
			"f1_macro": f1_score(y_test, prediction, average="macro"),
		}
		pipelines[name] = model
		predictions[name] = prediction
	prevalence = wmean(y_train.astype(float), w_train)
	results["Weighted majority baseline"] = {
		"n_train": len(y_train), "n_test": len(y_test),
		"weighted_accuracy": max(prevalence, 1 - prevalence),
		"weighted_balanced_accuracy": 0.5, "weighted_roc_auc": 0.5, "f1_macro": np.nan,
	}
	return results, len(sample), pipelines, (y_test, predictions)


mode_models, mode_model_n, mode_pipelines, (mode_y_test, mode_predictions) = fit_mode_model(data)
mode_model_table = pd.DataFrame(mode_models).T
mode_model_table.to_csv(OUT / "model_metrics_mode.csv", index_label="model")
display(mode_model_table)

In [ ]:
mode_importance = feature_importance_plot(
	mode_pipelines["Random forest"],
	"RQ3 model: what predicts Formal Bank vs. OTC?",
	"rq3_feature_importance.png",
)
display(Image(filename=str(OUT / "rq3_feature_importance.png")))
mode_importance.head(10)

In [ ]:
mode_confusion = confusion_matrix_plot(
	mode_y_test, mode_predictions["Random forest"],
	labels=["Formal Bank", "OTC"],
	title="RQ3 model: confusion matrix (random forest, held-out)",
	filename="rq3_confusion_matrix.png",
)
display(Image(filename=str(OUT / "rq3_confusion_matrix.png")))
mode_confusion

## 8. Nutshell plot

A single landscape figure summarizing all three research questions, suitable for a slide or poster.

In [ ]:
def plot_nutshell(data: pd.DataFrame, savings_auc: float, mode_auc: float) -> None:
	fig, axes = plt.subplots(1, 3, figsize=(13.34, 7.5), dpi=144)
	fig.patch.set_facecolor("#f3f0e8")
	fig.suptitle("Overseas work, savings, and remittances", fontsize=22, weight="bold", y=0.975)
	fig.text(0.5, 0.91, "A snapshot from the PSA Survey on Overseas Filipinos 2024", ha="center", fontsize=12)

	regional = data.dropna(subset=["region_group", "RSWGT"]).groupby("region_group", observed=True).apply(
		lambda group: 100 * wmean(group["contract"].astype(float), group["RSWGT"]),
		include_groups=False,
	).sort_values()
	axes[0].barh(regional.index, regional.values, color="#176b63")
	axes[0].set(title="Contract OFW share", xlabel="Weighted share (%)", xlim=(0, 100))

	savings = data.dropna(subset=["marital_status", "savings_pct", "RSWGT"])
	marital = savings.groupby("marital_status", observed=True).apply(
		lambda group: wmean(group["savings_pct"], group["RSWGT"]), include_groups=False
	).reindex(MARITAL_ORDER)
	axes[1].bar(range(len(marital)), marital.values, color="#3278a3")
	axes[1].set_xticks(range(len(marital)), marital.index, rotation=12)
	axes[1].set(title="Remittance savings share", ylabel="Weighted mean (%)")

	modes = data.dropna(subset=["mode", "urban_rural"])
	table = pd.crosstab(modes["urban_rural"], modes["mode"], normalize="index").mul(100)
	table.reindex(index=["Urban", "Rural"], columns=REMIT_MODE_ORDER, fill_value=0).plot(
		kind="bar", stacked=True, ax=axes[2], color=["#176b63", "#dd8a35", "#6c7780"], legend=False
	)
	axes[2].set(title="Remittance channel", ylabel="Respondents (%)", xlabel="")
	axes[2].tick_params(axis="x", rotation=0)
	handles, labels = axes[2].get_legend_handles_labels()
	fig.legend(handles, labels, title="Mode", loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.19), frameon=False, fontsize=8)

	fig.text(0.5, 0.145, "Occupation and origin track contract classification; marital status and destination track savings allocation. Channels vary by location.", ha="center", fontsize=8)
	fig.text(0.5, 0.10, f"Savings model ROC-AUC: {savings_auc:.2f} | Channel model ROC-AUC: {mode_auc:.2f} | Association is not causation", ha="center", fontsize=10, weight="bold")
	fig.text(0.5, 0.055, "Household region / urban-rural are geography proxies; province is unavailable and OTC groups several non-bank, non-door-to-door modes.", ha="center", fontsize=8)
	for axis in axes:
		axis.spines[["top", "right"]].set_visible(False)
		axis.grid(axis="y", alpha=0.2)
		axis.set_axisbelow(True)
	fig.tight_layout(rect=[0.03, 0.22, 0.97, 0.87])
	fig.savefig(OUT / "nutshell.png", dpi=144, facecolor=fig.get_facecolor())
	fig.savefig(OUT / "nutshell.pdf", facecolor=fig.get_facecolor())
	plt.close(fig)


plot_nutshell(data, savings_models["Random forest"]["weighted_roc_auc"], mode_models["Random forest"]["weighted_roc_auc"])
display(Image(filename=str(OUT / "nutshell.png")))

## 9. Assemble and write the full report

Builds `outputs/eda_report.md`: project framing, the three hypothesis tests, both machine-learning models, limitations, and a figure index.

In [ ]:
reported_savings = int(data["savings_pct"].notna().sum())
reported_modes = int(data["mode"].notna().sum())
savings_mean = wmean(data["savings_pct"], data["RSWGT"])

marital_text = "| Marital status | Weighted mean savings (%) | n |\n|---|---:|---:|\n"
marital_text += "\n".join(
	f"| {row.marital_status} | {row.weighted_mean_pct:.2f} | {int(row.n)} |"
	for row in marital_summary.itertuples(index=False)
)
destination_text = "| Destination region | Weighted mean savings (%) | n |\n|---|---:|---:|\n"
destination_text += "\n".join(
	f"| {row.destination_region} | {row.weighted_mean_pct:.2f} | {int(row.n)} |"
	for row in destination_summary.itertuples(index=False)
)
savings_model_text = "\n".join(
	f"| {name} | {m['weighted_accuracy']:.3f} | {m['weighted_balanced_accuracy']:.3f} | {m['weighted_roc_auc']:.3f} |"
	for name, m in savings_models.items()
)
mode_model_text = "\n".join(
	f"| {name} | {m['weighted_accuracy']:.3f} | {m['weighted_balanced_accuracy']:.3f} | {m['weighted_roc_auc']:.3f} |"
	for name, m in mode_models.items()
)

In [ ]:
report = f"""# Leaving for a Living: Labor Inequality and Financial Mobility of OFWs

**SDGs:** SDG 1 (No Poverty — poverty alleviation, income equality, social protection) and SDG 8 (Decent Work and Economic Growth — job creation, fair wages, economic development).

## Dataset and scope

- Source: PSA Survey on Overseas Filipinos 2024 Public Use File supplied in this repository.
- Records: {len(data):,}; distinct households: {data['RHHNUM'].nunique():,}. Household identifiers were used to cluster hypothesis-test standard errors and to keep households together in model evaluation.
- Survey year: {int(data['RSVYYR'].dropna().mode().iloc[0])}. This is not 2025/2026 data. The course guide permits 2024 only if the group documents that it is the latest available; verify and cite that status.
- `RSWGT` is used for descriptive estimates and model fitting/evaluation. Inferential models use unweighted observations with household-cluster robust covariance; the PUF does not provide enough survey-design details for fully design-based standard errors.
- Blanks are missing; zero savings is a valid answer. Savings allocation is reported for {reported_savings:,} records. Remittance mode is reported for {reported_modes:,}; door-to-door is only {door_count} records.

## RQ1: what drives OFWs toward contractual overseas work

**H0:** Pre-departure domestic employment characteristics and regional demographics have no statistical significance on why OFWs migrate to contractual overseas work.
**H1:** OFWs come disproportionately from lower-wage domestic occupations and from outside the capital region, which gives statistical significance to migrating for contractual labor.

Operationalization: contract-OFW status (`RQ10_REASON = 1`) among people already represented in the overseas-Filipino survey, compared across usual occupation and NCR/outside-NCR household origin. Both are genuinely pre-departure per the questionnaire (C9 occupation "before he/she (last) left"), but the file still does not include non-migrants or domestic wages; occupation is only a prior-work proxy and region is only a rough inequality proxy, so this tests association among surveyed OFWs rather than a true push-factor model.

- Weighted contract-OFW share: **{contract_share:.1f}%**.
- Occupation, adjusted for NCR/outside NCR: household-cluster robust linear probability model, Wald F({h1_occupation['df']})={h1_occupation['statistic']:.2f}, p {pvalue(h1_occupation['pvalue'])}, N={len(h1_data):,}; crude Cramer's V={h1_crude_occupation['cramers_v']:.3f}.
- NCR vs outside NCR, adjusted for occupation: household-cluster robust Wald F({h1_location['df']})={h1_location['statistic']:.2f}, p {pvalue(h1_location['pvalue'])}; crude Cramer's V={h1_crude_location['cramers_v']:.3f}.

The results support or reject association among surveyed overseas Filipinos; they cannot establish why people migrate or whether lower domestic wages cause contractual migration, since no non-migrant comparison group is available.

## RQ2: demographics and maintaining personal savings abroad

**H0:** An individual's demographic characteristics have no statistically significant relationship with an OFW's allocation of monthly income with regards to personal savings.
**H1:** Having a higher level of educational attainment and a secure contract status make it more likely an OFW maintains personal savings rather than remitting nearly all earnings to their family.

Operationalization: the questionnaire's C18 asks exactly what RQ2 needs (personal savings vs. remittance share of monthly compensation abroad), but `RQ18_*` is absent from the released PUF (the dictionary jumps from C17 to C19). `RQ25_CASHAMT` (C25) also can't substitute — it is a fixed six-month sum (April–Sept 2024) that does not line up with `RQ19_NOMONTH` (months worked over the full five-year window). The closest available proxy is whether any share of what *is* remitted is allocated to savings by the receiving family (`RQ273P_SAVINGS > 0`, `any_savings`, from C27) versus all of it going to consumption/gifts. This is tested against the predictors named in the research question — sex, marital status, destination country (grouped into regions), and contract status — with household-clustered robust standard errors.

- Any-savings share among surveyed OFWs: **{any_savings_share:.1f}%**.
- Sex, adjusted for marital status, destination, and contract: Wald F({h2_sex['df']})={h2_sex['statistic']:.2f}, p {pvalue(h2_sex['pvalue'])}, N={len(savings):,}; crude Cramer's V={h2_crude_sex['cramers_v']:.3f}.
- Marital status, adjusted for the above: Wald F({h2_marital['df']})={h2_marital['statistic']:.2f}, p {pvalue(h2_marital['pvalue'])}; crude Cramer's V={h2_crude_marital['cramers_v']:.3f}.
- Destination region, adjusted for the above: Wald F({h2_destination['df']})={h2_destination['statistic']:.2f}, p {pvalue(h2_destination['pvalue'])}; crude Cramer's V={h2_crude_destination['cramers_v']:.3f}.
- Contract status, adjusted for the above: Wald F({h2_contract['df']})={h2_contract['statistic']:.2f}, p {pvalue(h2_contract['pvalue'])}; crude Cramer's V={h2_crude_contract['cramers_v']:.3f}.

Weighted savings means by marital status:

{marital_text}

Weighted savings means by destination region:

{destination_text}

## RQ3: remittance transfer mode and regional location

**H0:** The choice of remittance transfer mode is statistically independent of regional location.
**H1:** Recipient households in island and rural areas rely more on OTC or door-to-door transfer compared to urbanized locations.

Operationalization: `RQ26_MODE` is recoded into three channels matching the hypothesis — **Formal Bank**, **Door-to-door**, and **OTC** (agency/local office, friends/co-workers, money-transfer services, and other modes). Geography available is household region (island group) and urban/rural classification, not province.

- Urban/rural terms: household-cluster robust multinomial Wald chi-square({h3_tests['Urban/rural location']['df']})={h3_tests['Urban/rural location']['statistic']:.2f}, p {pvalue(h3_tests['Urban/rural location']['pvalue'])}.
- Island-group terms: household-cluster robust multinomial Wald chi-square({h3_tests['Island group']['df']})={h3_tests['Island group']['statistic']:.2f}, p {pvalue(h3_tests['Island group']['pvalue'])}.
- All location terms jointly: Wald chi-square({h3_tests['All location terms']['df']})={h3_tests['All location terms']['statistic']:.2f}, p {pvalue(h3_tests['All location terms']['pvalue'])}; crude Cramer's V={h3_urban['cramers_v']:.3f} for urban/rural and {h3_region['cramers_v']:.3f} for island group.

Door-to-door has only {door_count} observations, so any door-to-door-specific conclusion is not reliable; OTC is overwhelmingly driven by formal money-transfer services rather than informal carriers.

## Machine learning (objectives 2 and 3)

**Objective 2 model — predicting any reported savings.** Predictors: demographic traits (age, sex, marital status), human capital (highest grade completed), contract status, and foreign destination (destination region). Household-grouped stratified 80/20 split (seed 132), survey-weighted fitting and scoring. Evaluation sample: {savings_model_n:,}.

| Model | Weighted accuracy | Weighted balanced accuracy | Weighted ROC-AUC |
|---|---:|---:|---:|
{savings_model_text}

**Objective 3 model — classifying Formal Bank vs. OTC.** Predictors: island group, urban/rural (geography), and monthly income, education, age (socioeconomic status). Door-to-door excluded ({door_count} records, insufficient for train/test splitting). Evaluation sample: {mode_model_n:,}.

| Model | Weighted accuracy | Weighted balanced accuracy | Weighted ROC-AUC |
|---|---:|---:|---:|
{mode_model_text}

Both models beat their weighted-majority baselines but leave most variance unexplained — treat them as supporting evidence for the determinant groups named in the objectives, not as causal or production-ready classifiers.

## Limitations and next steps

1. The supplied file is from 2024. Document that it is the latest available for this survey if using the course's 2024-data exception.
2. There is no non-migrant comparison group, pre-departure wage field, province identifier, or contract-security measure.
3. The questionnaire's C18 (personal savings vs. remittance allocation of monthly compensation abroad — the field RQ2 names directly) was collected but not released in this PUF; `any_savings` (share of the *remitted* amount the family allocates to savings, from C27) is the closest available substitute, and over one-quarter of records do not report it.
4. OTC is not a single survey category; it is a grouping of agency/local-office, friends/co-workers, money-transfer-service, and other codes, dominated by money-transfer services. Door-to-door is reported only {door_count} times, so it is excluded from the RQ3 classifier entirely.
5. `RQ24`–`RQ31` (remittance amount, mode, spending shares) are only asked of respondents flagged `RP6M_IND == 1`; this is a questionnaire skip pattern, not random missingness, and it is why RQ2/RQ3 run on roughly 2,600–2,700 records versus {len(data):,} total.
6. The file has survey weights, but inferential models are not survey-design based because strata/PSU information is not supplied. Cluster-robust standard errors address repeated household records only; interpret p-values cautiously.

## Figures

- `h1_contract_status.png`, `h2_savings_allocation.png`, `h3_mode_by_location.png`
- `rq2_feature_importance.png`, `rq2_confusion_matrix.png`
- `rq3_feature_importance.png`, `rq3_confusion_matrix.png`
- `nutshell.png` / `nutshell.pdf` (1920 x 1080 landscape)
"""
(OUT / "eda_report.md").write_text(report, encoding="utf-8")
print(report)

## 10. Save hypothesis-test and model-metrics tables

In [ ]:
tests = [
	{"test": "H1 occupation adjusted for origin", "n": len(h1_data), **h1_occupation},
	{"test": "H1 NCR origin adjusted for occupation", "n": len(h1_data), **h1_location},
	{"test": "H2 sex adjusted for marital/destination/contract", "n": len(savings), **h2_sex},
	{"test": "H2 marital status adjusted for sex/destination/contract", "n": len(savings), **h2_marital},
	{"test": "H2 destination region adjusted for sex/marital/contract", "n": len(savings), **h2_destination},
	{"test": "H2 contract adjusted for sex/marital/destination", "n": len(savings), **h2_contract},
	{"test": "H3 urban-rural location terms", "n": len(modes), **h3_tests["Urban/rural location"]},
	{"test": "H3 island group terms", "n": len(modes), **h3_tests["Island group"]},
	{"test": "H3 all location terms jointly", "n": len(modes), **h3_tests["All location terms"]},
]
pd.DataFrame(tests).to_csv(OUT / "hypothesis_tests.csv", index=False)
print(f"Analysis complete: {OUT}")
print(f"Records: {len(data):,}; households: {data['RHHNUM'].nunique():,}")
print(f"Savings reported: {reported_savings:,}; modes reported: {reported_modes:,}")
display(pd.DataFrame(tests))

## 11. Key findings

- **RQ1:** Contract-OFW status is strongly associated with prior occupation (p < 0.001) and, more weakly, with NCR-vs-outside-NCR household origin — consistent with H1, but only as association among already-surveyed OFWs, not a causal push-factor claim.
- **RQ2:** Sex and marital status are significantly associated with whether any remitted earnings are allocated to savings; destination region and contract status are not significant once the others are controlled for — partial support for H1, but not through the contract-security channel the hypothesis names.
- **RQ3:** Urban/rural and island-group location are both strongly associated with remittance channel — rural and Mindanao households lean much more heavily on OTC than NCR/urban households, supporting H1. Door-to-door is too rare in this file to evaluate on its own.
- **ML (objective 2):** A random forest predicts any-savings with modest lift over the majority baseline (see ROC-AUC above); sex, marital status, and destination region are the strongest drivers.
- **ML (objective 3):** A random forest separates Formal Bank from OTC users well above baseline using only geography and socioeconomic status, directly supporting objective 3's financial-access framing.
- **Overall:** findings are associational, drawn from a single cross-sectional PUF with no non-migrant comparison group — see the Limitations section of `outputs/eda_report.md` before drawing policy conclusions.